# Task 3: Financial Research Multi-Agent System — Final Demonstration

This notebook provides a comprehensive, step-by-step verification and demonstration of **Task 3: Financial Research Agent System** built using **LangChain** and **LangGraph**.

### 📋 Evaluator Section Index
1. [Project Setup & Configuration](#1.-Project-Setup-&-Configuration)
2. [Five Independent Tools Demonstration](#2.-Five-Independent-Tools-Demonstration)
3. [Task 3A: Single Autonomous Agent Demonstration](#3.-Task-3A:-Single-Autonomous-Agent-Demonstration)
4. [Autonomous Tool-Selection Trace](#4.-Autonomous-Tool-Selection-Trace)
5. [Observe -> Replan -> Act State Transition Trace](#5.-Observe----Replan----Act-State-Transition-Trace)
6. [Final Task 3A Research Report](#6.-Final-Task-3A-Research-Report)
7. [Task 3B: Agent A (Quantitative Data Analyst) Demonstration](#7.-Task-3B:-Agent-A-(Quantitative-Data-Analyst)-Demonstration)
8. [DataBrief Structured Handoff Payload](#8.-DataBrief-Structured-Handoff-Payload)
9. [Task 3B: Agent B (Qualitative Research Writer) Demonstration](#9.-Task-3B:-Agent-B-(Qualitative-Research-Writer)-Demonstration)
10. [Tool Access Control & Restriction Evidence](#10.-Tool-Access-Control-&-Restriction-Evidence)
11. [Task 3B Critique / Clarification Loop Demonstration](#11.-Task-3B-Critique-/-Clarification-Loop-Demonstration)
12. [Final Multi-Agent Research Report](#12.-Final-Multi-Agent-Research-Report)
13. [Short-Term Memory Demonstration (Multi-Turn Thread Session)](#13.-Short-Term-Memory-Demonstration-(Multi-Turn-Thread-Session))
14. [Persistent Memory Demonstration (JSON Disk Cache Hit)](#14.-Persistent-Memory-Demonstration-(JSON-Disk-Cache-Hit))
15. [Persistent Execution Audit Log (`agent_trace.jsonl`)](#15.-Persistent-Execution-Audit-Log-(agent_trace.jsonl))
16. [Assessment Requirement-to-Evidence Matrix](#16.-Assessment-Requirement-to-Evidence-Matrix)

## 1. Project Setup & Configuration

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Initialize notebook imports, path resolvers, and config loading via python-dotenv', Date: 2026-09-11
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import config

print(f"Project Base Directory: {config.BASE_DIR}")
print(f"LLM Provider:           {config.LLM_PROVIDER}")
print(f"LLM Model Name:         {config.LLM_MODEL_NAME}")
print(f"Cache Directory:        {config.CACHE_DIR}")
print(f"Trace File Path:        {config.TRACE_FILE}")

## 2. Five Independent Tools Demonstration

Demonstrating independent invocation of all five LangChain research tools (`get_price_data`, `get_news`, `calculate_volatility`, `llm_sentiment`, `web_search`).

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Demonstrate individual tool calls for price data, news, volatility, LLM sentiment, and web search', Date: 2026-09-11
from src.tools import get_price_data, get_news, calculate_volatility, llm_sentiment, web_search

TICKER = "AAPL"

print("1. get_price_data tool invocation:")
res_price = get_price_data.invoke({"ticker": TICKER, "period": "1mo"})
print(f"   Status: {res_price['status']} | Count: {res_price['records_count']} | Latest Close: ${res_price['latest_indicators']['latest_close']}")

print("\n2. calculate_volatility tool invocation:")
res_vol = calculate_volatility.invoke({"ticker": TICKER, "window": 60})
print(f"   Status: {res_vol['status']} | Annualized Volatility: {res_vol['annualized_volatility']}%")

print("\n3. get_news tool invocation:")
res_news = get_news.invoke({"ticker": TICKER, "n": 3})
print(f"   Status: {res_news['status']} | Headlines Retrieved: {res_news['count']}")

print("\n4. llm_sentiment tool invocation:")
headlines_sample = [item['title'] for item in res_news.get('news', [])[:2]] or ["Apple stock rises on earnings beat"]
res_sent = llm_sentiment.invoke({"headlines": headlines_sample})
print(f"   Status: {res_sent['status']} | Label: {res_sent['sentiment']['label']} | Score: {res_sent['sentiment']['score']}")

print("\n5. web_search tool invocation:")
res_search = web_search.invoke({"query": f"{TICKER} stock analyst price targets", "max_results": 2})
print(f"   Status: {res_search['status']} | Search Results Count: {res_search['count']}")

## 3. Task 3A: Single Autonomous Agent Demonstration

Executing Task 3A single autonomous agent (`run_financial_research_agent`).

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Execute Task 3A single autonomous research agent workflow for AAPL', Date: 2026-09-11
from src.workflows import run_financial_research_agent

print(f"===========================================================================")
print(f"🚀 RUNNING TASK 3A SINGLE AUTONOMOUS AGENT FOR: {TICKER}")
print(f"===========================================================================\n")

task3a_state = run_financial_research_agent(ticker=TICKER, use_cache=False)

## 4. Autonomous Tool-Selection Trace

Demonstrating that the agent's tool execution order is **not hard-coded**, but decided dynamically at runtime by the LLM based on current state context.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Display tool selection trace steps showing autonomous tool choices', Date: 2026-09-11
print("Executed Tool Calls Trajectory (Autonomous Selection):")
for msg in task3a_state['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  -> Tool Selected: {tc['name']:<22} | Arguments: {tc['args']}")

## 5. Observe -> Replan -> Act State Transition Trace

Demonstrating the explicit 5-step state transition loop: **Agent decision $\rightarrow$ Tool call $\rightarrow$ Tool result $\rightarrow$ Updated observation $\rightarrow$ New decision**.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Display structured observation records stored in graph state', Date: 2026-09-11
print("Recorded Graph State Observations (Observe -> Replan -> Act):\n")
for idx, obs in enumerate(task3a_state.get('observations', []), 1):
    print(f"Observation #{idx}:")
    print(f"  Tool Name:  {obs.get('tool_name')}")
    print(f"  Status:     {obs.get('status')}")
    print(f"  Summary:    {obs.get('summary')}\n")

## 6. Final Task 3A Research Report

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Print synthesized Task 3A final equity research report', Date: 2026-09-11
print("📄 TASK 3A FINAL RESEARCH REPORT:\n")
print(task3a_state.get('final_report', 'No report generated.'))

## 7. Task 3B: Agent A (Quantitative Data Analyst) Demonstration

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Execute Task 3B Agent A Data Analyst workflow for AAPL', Date: 2026-09-11
from src.workflows import run_data_analyst_agent

print(f"===========================================================================")
print(f"🚀 RUNNING TASK 3B AGENT A (DATA ANALYST) FOR: {TICKER}")
print(f"===========================================================================\n")

agent_a_state = run_data_analyst_agent(ticker=TICKER)

## 8. DataBrief Structured Handoff Payload

Displaying the strongly-typed Pydantic `DataBrief` payload saved into `state["data_brief"]` ready for downstream consumption by Agent B.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Display structured DataBrief handoff dictionary', Date: 2026-09-11
print("🤝 STRUCTURED DATABRIEF HANDOFF PAYLOAD (POPULATED IN GRAPH STATE):\n")
data_brief_payload = agent_a_state.get("data_brief")
print(json.dumps(data_brief_payload, indent=2))

## 9. Task 3B: Agent B (Qualitative Research Writer) Demonstration

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Execute Task 3B Multi-Agent sequential workflow (Agent A -> Handoff -> Agent B)', Date: 2026-09-11
from src.workflows import run_multi_agent_research

print(f"===========================================================================")
print(f"🚀 RUNNING MULTI-AGENT WORKFLOW (AGENT A -> HANDOFF -> AGENT B) FOR: {TICKER}")
print(f"===========================================================================\n")

multi_agent_state = run_multi_agent_research(ticker=TICKER, use_cache=False)

## 10. Tool Access Control & Restriction Evidence

Demonstrating code-level enforcement of tool restrictions for sub-agents.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Display tool access control matrix for Agent A and Agent B', Date: 2026-09-11
from src.agents import DATA_ANALYST_TOOLS, RESEARCH_WRITER_TOOLS

print("Tool Access Control Security Matrix:")
print(f"  Agent A (Data Analyst) Allowed Tools:     {[t.name for t in DATA_ANALYST_TOOLS]}")
print(f"  Agent A Prohibited Tools:                  ['get_news', 'web_search']")
print(f"  Agent B (Research Writer) Allowed Tools:   {[t.name for t in RESEARCH_WRITER_TOOLS]}")
print(f"  Agent B Prohibited Tools:                  ['get_price_data', 'calculate_volatility']")

## 11. Task 3B Critique / Clarification Loop Demonstration

Displaying the structured Pydantic `ClarificationRequest` issued by Agent B, the numerical `ClarificationResponse` returned by Agent A, and the single-loop recursion guard.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Display ClarificationRequest and ClarificationResponse state payloads', Date: 2026-09-11
print("❓ ClarificationRequest Issued by Agent B to Agent A:")
print(json.dumps(multi_agent_state.get('clarification_request'), indent=2))

print("\n💡 ClarificationResponse Calculated & Returned by Agent A:")
print(json.dumps(multi_agent_state.get('clarification_response'), indent=2))

print(f"\nLoop Guard Enforced: clarification_count = {multi_agent_state.get('clarification_count')} (Max 1 loop guard active)")

## 12. Final Multi-Agent Research Report

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Print final synthesized multi-agent research report', Date: 2026-09-11
print("📄 FINAL MULTI-AGENT RESEARCH REPORT (PRODUCED BY AGENT B):\n")
print(multi_agent_state.get('final_report', 'No report generated.'))

## 13. Short-Term Memory Demonstration (Multi-Turn Thread Session)

Demonstrating multi-turn thread session memory using LangGraph `MemorySaver`. A follow-up question answers directly from state memory without re-calling tools.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Demonstrate multi-turn thread session follow-up using short-term state memory', Date: 2026-09-11
SESSION_THREAD = "session_aapl_demo_101"

# Turn 1: Main Research Execution
turn_1 = run_multi_agent_research(ticker=TICKER, question=f"Analyse {TICKER} financial health", use_cache=False, thread_id=SESSION_THREAD)

# Turn 2: Follow-up question answering directly from state memory without tools
turn_2 = run_multi_agent_research(ticker=TICKER, question=f"What was the 14-day RSI and 252-day volatility you calculated for {TICKER}?", use_cache=False, thread_id=SESSION_THREAD)

print("Follow-up Response (Answered from Short-Term Graph State Memory):\n")
print(turn_2.get('final_report'))

## 14. Persistent Memory Demonstration (JSON Disk Cache Hit)

Demonstrating deterministic disk caching (`cache/{TICKER}_{YYYY-MM-DD}.json`). Subsequent calls load instantly from cache, bypassing API calls.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Demonstrate persistent disk cache hit and schema validation', Date: 2026-09-11
from src.memory import load_persistent_cache, get_cache_file_path

# Check persistent cache file
cache_path = get_cache_file_path(TICKER)
print(f"Cache File Path: {cache_path}")
print(f"Cache File Exists: {cache_path.exists()}")

# Call run_multi_agent_research with use_cache=True (Instant Cache Hit)
cached_result = run_multi_agent_research(ticker=TICKER, use_cache=True)
print(f"\nPersistent Cache Hit Status: Success | Cached Timestamp: {cached_result.get('timestamp')}")

## 15. Persistent Execution Audit Log (`agent_trace.jsonl`)

Displaying persistent trace log entries from `agent_trace.jsonl` recording `AGENT`, `TOOL CALL`, `TOOL RESULT`, `UPDATED OBSERVATION`, `AGENT DECISION`, and `HANDOFF` events.

In [ ]:
# AI-ASSISTED: Gemini (gemini-3.6-flash), Prompt: 'Read and display sample audit entries from agent_trace.jsonl', Date: 2026-09-11
trace_file = config.TRACE_FILE
if trace_file.exists():
    with open(trace_file, 'r', encoding='utf-8') as f:
        lines = [json.loads(line) for line in f.readlines() if line.strip()]
    print(f"Total Execution Audit Trace Records Logged: {len(lines)}")
    print("\nSample Logged Events (Last 5 Entries):")
    for entry in lines[-5:]:
        print(f"  [{entry.get('event_type'):<20}] {entry.get('content')[:90]}...")
else:
    print("Trace file not found.")

## 16. Assessment Requirement-to-Evidence Matrix

| Assessment Requirement | Code Location / Implementation | Notebook Verification Section |
| :--- | :--- | :--- |
| **Project Foundation & Config** | [src/config.py](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/config.py) | [Section 1: Project Setup](#1.-Project-Setup-&-Configuration) |
| **5 Independent Tools** | [src/tools/](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/tools/) | [Section 2: Tool Demos](#2.-Five-Independent-Tools-Demonstration) |
| **LangChain Tool Integration** | `@tool` & `args_schema` in [src/tools/](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/tools/) | [Section 2: Tool Demos](#2.-Five-Independent-Tools-Demonstration) |
| **Task 3A Autonomous Agent** | [src/agents/research_agent.py](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/agents/research_agent.py) | [Section 3: Task 3A Single Agent](#3.-Task-3A:-Single-Autonomous-Agent-Demonstration) |
| **Autonomous Tool Selection** | Dynamic LLM routing (no hardcoded order) | [Section 4: Selection Trace](#4.-Autonomous-Tool-Selection-Trace) |
| **Observe-Replan-Act Cycle** | `AgentState["observations"]` channel | [Section 5: Observe-Replan Trace](#5.-Observe----Replan----Act-State-Transition-Trace) |
| **Task 3A Final Report** | Financial Health, 3 Risks, Hedge Strategy | [Section 6: Task 3A Report](#6.-Final-Task-3A-Research-Report) |
| **Task 3B Agent A (Data Analyst)** | `DATA_ANALYST_TOOLS` in [src/agents/data_analyst_agent.py](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/agents/data_analyst_agent.py) | [Section 7: Agent A Demo](#7.-Task-3B:-Agent-A-(Quantitative-Data-Analyst)-Demonstration) |
| **DataBrief Structured Handoff** | Pydantic `DataBrief` in `state["data_brief"]` | [Section 8: DataBrief Handoff](#8.-DataBrief-Structured-Handoff-Payload) |
| **Task 3B Agent B (Research Writer)** | `RESEARCH_WRITER_TOOLS` in [src/agents/research_writer_agent.py](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/src/agents/research_writer_agent.py) | [Section 9: Agent B Demo](#9.-Task-3B:-Agent-B-(Qualitative-Research-Writer)-Demonstration) |
| **Tool Restriction Control** | Code-level tool list isolation | [Section 10: Tool Restriction](#10.-Tool-Access-Control-&-Restriction-Evidence) |
| **Critique / Clarification Loop** | `ClarificationRequest` & `ClarificationResponse` | [Section 11: Clarification Loop](#11.-Task-3B-Critique-/-Clarification-Loop-Demonstration) |
| **Final Multi-Agent Report** | Comprehensive synthesized report | [Section 12: Final Multi-Agent Report](#12.-Final-Multi-Agent-Research-Report) |
| **Short-Term Memory** | `MemorySaver` checkpointer thread state | [Section 13: Short-Term Memory](#13.-Short-Term-Memory-Demonstration-(Multi-Turn-Thread-Session)) |
| **Persistent Memory** | `cache/{TICKER}_{YYYY-MM-DD}.json` | [Section 14: Persistent Cache Hit](#14.-Persistent-Memory-Demonstration-(JSON-Disk-Cache-Hit)) |
| **Audit Trace Logging** | `agent_trace.jsonl` persistence | [Section 15: Audit Trace Log](#15.-Persistent-Execution-Audit-Log-(agent_trace.jsonl)) |
| **AI Assistance Citations** | `CITATIONS.md` & code headers | [CITATIONS.md](file:///c:/Users/mskan/Desktop/Technical_Assyment_Cdazzdev/task3_financial_agents/CITATIONS.md) |